In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/tp4_dasd/configs/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
%pip install -e .[torch,bitsandbytes]
%pip install -U bitsandbytes>=0.46.1

In [ ]:
%cd /content/drive/MyDrive/tp4_dasd/configs/
%ls

!llamafactory-cli train --stage sft --do_train --model_name_or_path unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit --dataset train_low_temp_student.json --template qwen3_nothink --finetuning_type lora --lora_rank 8 --lora_target all --output_dir /content/drive/MyDrive/tp4_dasd/outputs/stage1 --overwrite_output_dir --plot_loss --trust_remote_code --per_device_train_batch_size 1 --gradient_accumulation_steps 8 --learning_rate 1.0e-4 --num_train_epochs 3.0 --lr_scheduler_type cosine --warmup_ratio 0.1 --logging_steps 10 --save_steps 500 --cutoff_len 2048 --max_samples 1000 --preprocessing_num_workers 16 --dataloader_num_workers 4 --fp16 --report_to none

In [ ]:
import json
import numpy as np
import openai
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configuration API (gardée ici pour clonage local par thread)
API_BASE_URL = "https://api.infomaniak.com/2/ai/48/openai/v1"
API_KEY = "nKuJabWS1epvq3x-m8by6NOU4xP4_znNL9OhmgXBPz9OeWOHlyGJIENnG8oXLT-4oOXNmESqExEMZv6o"
TEACHER_MODEL = "openai/gpt-oss-120b"
BLOC_SIZE = 1
TOTAL_THREADS = 10
LOW_TEMP = 0.15
HIGH_TEMP = 0.9

# Résultats globaux
responses = []

# Chargement des données une seule fois (mémoire)
with open("/content/drive/MyDrive/tp4_dasd/data/result.json", "r") as f:
    data = json.load(f)

def process_block(block_idx, start_idx, end_idx, temperature):
    """Traite un bloc de puzzles (start_idx..end_idx-1) avec la température fournie."""
    local_responses = []
    # Client par thread pour éviter des problèmes de concurrence
    client = openai.OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
    for i in range(start_idx, end_idx):
        try:
            puzzle = data[i]
            puzzle_id = puzzle["puzzle_id"]
            puzzle_initial_fen = puzzle["initial_fen"]
            puzzle_ennemy_moves = puzzle["solution_san"][1::2]
            puzzle_nb_moves = len(puzzle_ennemy_moves) + 1
            print(f"Block {block_idx} - Puzzle {i} [ID:{puzzle_id}] - FEN: {puzzle_initial_fen} - N: {puzzle_nb_moves}")

            response = client.chat.completions.create(
                model=TEACHER_MODEL,
                logprobs=True,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": "You are a chess expert and you'll try to answer chess puzzles. You'll be provided with a position in SAN format and should find the best N moves from that position. N is provided by the user."},
                    {"role": "user", "content": "I have a chess puzzle for you. The initial position is: " + puzzle_initial_fen + ". Try to find a mate or stalemate in " + str(puzzle_nb_moves) + " moves in SAN format. The ennemy will do these moves: " + "/".join(puzzle_ennemy_moves) + ". Return only the moves in SAN format, without any explanation or commentary. If you can't find a solution, return 'No solution found'."},
                ],
                max_tokens=8192,
            )

            # Extraction des logprobs (si fournis)
            logprobs_data = response.choices[0].logprobs
            tokens = []
            logprobs = []
            if logprobs_data:
                for token_info in logprobs_data.content:
                    tokens.append(token_info.token)
                    logprobs.append(token_info.logprob)

            total_logprob = sum(logprobs) if logprobs else 0.0
            mean_logprob = np.exp(np.mean(logprobs)) if logprobs else 0.0

            local_responses.append({
                "puzzle": puzzle_id,
                "content": response.choices[0].message.content,
                "log_prob": mean_logprob,
                "block": block_idx,
                "temperature": temperature
            })
            print(f"Block {block_idx} - Puzzle {i} [ID:{puzzle_id}] - LogProb: {mean_logprob:.4f}")
        except Exception as e:
            print(f"Erreur bloc {block_idx} puzzle {i}: {e}")
    return local_responses

blocks = []
for b in range(TOTAL_THREADS):
    start = b * BLOC_SIZE
    end = start + BLOC_SIZE
    temp = LOW_TEMP if b < (TOTAL_THREADS // 2) else HIGH_TEMP
    blocks.append((b, start, end, temp))

# Exécution parallèle (ThreadPoolExecutor convient pour les appels I/O réseau)
with ThreadPoolExecutor(max_workers=TOTAL_THREADS) as ex:
    futures = [ex.submit(process_block, b, s, e, t) for (b,s,e,t) in blocks]
    for fut in as_completed(futures):
        try:
            res = fut.result()
            responses.extend(res)
        except Exception as e:
            print("Un bloc a échoué :", e)

# Enregistrement des réponses agrégées
with open("/content/drive/MyDrive/tp4_dasd/outputs/stage1/responses.json", "w") as f:
    json.dump(responses, f, indent=4)

In [ ]:
import numpy as np
import torch
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


class DASPipelineQwen:
    def __init__(self, openai_api_key, student_model_id="unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"):
        """
        Initialise le pipeline DAS avec un Teacher (API) et un Student (Local 4-bit).
        """
        # 1. Configuration Student (4-bit quantization)
        bnb_config = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, )

        self.student_model_id = student_model_id
        print(f"Chargement du modèle étudiant : {self.student_model_id}...")

        self.tokenizer = AutoTokenizer.from_pretrained(self.student_model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
                self.student_model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
                )
        self.model.eval()

        # 2. Configuration Teacher (OpenAI Compatible API - ex: Infomaniak)
        # Note: Remplacez base_url par l'URL correcte si différent de l'exemple
        self.client = OpenAI(
                api_key=openai_api_key, base_url="https://api.infomaniak.com/2/ai/48/openai/v1"
                )
        self.teacher_model_name = "openai/gpt-oss-120b"

    def get_teacher_data(self, user_prompt, temperature=0.7):
        """
        Génère la réponse du Teacher avec les logprobs.
        """
        messages = [{"role": "user", "content": user_prompt}]
        # Note: Assurez-vous que le modèle supporte logprobs=True
        response = self.client.chat.completions.create(
                model=self.teacher_model_name, messages=messages, temperature=temperature, logprobs=True, top_logprobs=1
                )

        content = response.choices[0].message.content
        logprobs_data = response.choices[0].logprobs
        tokens = []
        logprobs = []
        # On vérifie si logprobs est disponible (certaines API compatibles ne le renvoient pas)
        if logprobs_data:
            for token_info in logprobs_data.content:
                tokens.append(token_info.token)
                logprobs.append(token_info.logprob)
        else:
            raise ValueError("L'API Teacher n'a pas renvoyé de logprobs. Vérifiez la compatibilité.")

        # Compute total log probability (sum of logprobs)
        total_logprob = sum(logprobs) if logprobs else 0.0

        # Compute geometric mean of probabilities
        # P_geom = exp(mean(logprobs))
        mean_logprob = np.exp(np.mean(logprobs)) if logprobs else 0.0
        return {
            "content":      content, "tokens": tokens, "logprobs": logprobs, "total_logprob": total_logprob,
            "mean_logprob": mean_logprob, "num_tokens": len(tokens)
            }

    def get_student_logprobs(self, prompt: str, response: str) -> dict:
        """
        Calcule les log-probabilités de la réponse (Student) de manière robuste.
        Utilise la méthode de masquage standard (Labels = -100 pour le prompt).
        """
        # 1. Préparer le texte complet (Prompt + Réponse)
        # On utilise le chat template qui gère proprement les balises <|im_start|>, etc.
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response}
            ]
        full_text = self.tokenizer.apply_chat_template(messages, tokenize=False)

        # 2. Tokenizer le tout
        # return_tensors='pt' nous donne directement les tenseurs PyTorch
        inputs = self.tokenizer(full_text, return_tensors="pt").to(self.model.device)
        input_ids = inputs.input_ids

        # 3. Identifier la longueur du Prompt pour le masquage
        # On regénère le prompt SEUL avec l'amorce de réponse (add_generation_prompt=True)
        # Cela inclut "<|im_start|>assistant\n" à la fin, pour s'aligner parfaitement.
        prompt_messages = [{"role": "user", "content": prompt}]
        prompt_text = self.tokenizer.apply_chat_template(
                prompt_messages, tokenize=False, add_generation_prompt=True
                )

        # On tokenise le prompt seul pour avoir sa longueur exacte en tokens
        prompt_tokens = self.tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids
        response_start_idx = prompt_tokens.shape[1]

        # 4. Créer les Labels (Masking du Prompt)
        # -100 est l'index ignoré par défaut par CrossEntropyLoss de PyTorch
        labels = input_ids.clone()
        # On masque tout ce qui est avant le début de la réponse
        labels[:, :response_start_idx] = -100

        # 5. Calcul "Clean" avec CrossEntropyLoss
        with torch.no_grad():
            outputs = self.model(input_ids)
            logits = outputs.logits

            # Shift des logits et labels pour la prédiction "next token"
            # logits[t] prédit labels[t+1]
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            # reduction='none' nous donne la perte pour chaque token individuel
            loss_fct = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
            token_losses = loss_fct(shift_logits.transpose(1, 2), shift_labels)

            # La Loss est par définition -log(p), donc log_prob = -loss
            token_logprobs = -token_losses

            # On ne garde que les tokens de la réponse (ceux qui n'étaient pas masqués à -100)
            # Note: shift_labels a été décalé, donc on utilise son masque
            valid_mask = shift_labels != -100
            valid_logprobs = token_logprobs[valid_mask].cpu().numpy()

        # Calcul des statistiques DAS
        total_logprob = np.sum(valid_logprobs)
        mean_logprob = np.exp(np.mean(valid_logprobs)) if len(valid_logprobs) > 0 else 0.0

        return {
            "total_logprob": total_logprob,
            "mean_logprob":  mean_logprob,
            "num_tokens":    len(valid_logprobs),
            "logprobs":      valid_logprobs.tolist()
            }

    def decide_keep_prompt(self, teacher_answer, student_answer):
        teacher_logprob = teacher_answer.get("mean_logprob", 0.0)
        student_logprob = student_answer.get("mean_logprob", 0.0)

        print(teacher_logprob, student_logprob)

        divergence = teacher_logprob - student_logprob

        print(divergence)

    def run(self, prompt):
        print(f"Traitement du prompt : '{prompt}'")

        # 1. Teacher
        teacher_answer = self.get_teacher_data(prompt)
        if not teacher_answer:
            return

        print(f"Réponse Teacher reçue ({len(teacher_answer["content"])} chars).")

        # 2. Student & Calculs
        try:
            student_answer = self.get_student_logprobs(prompt, teacher_answer["content"])

            # 3. Décision
            return self.decide_keep_prompt(teacher_answer, student_answer)

        except Exception as e:
            print(f"Erreur durant le calcul DAS : {e}")
            import traceback
            traceback.print_exc()
            return None


# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # Clé API
    API_KEY = "token here"

    # ID Modèle Étudiant (Compatible 4-bit unsloth/bnb)
    STUDENT_ID = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"

    pipeline = DASPipelineQwen(openai_api_key=API_KEY, student_model_id=STUDENT_ID)

    # Test
    test_prompt = "Explique le principe de la supraconductivité de manière simple."
    result = pipeline.run(test_prompt)